# OpenBind blind challenge — test run

Goal of this notebook: **time two predictions**, then decide if 400 is affordable.
Do not run the full set until the numbers here add up.

Runtime > Change runtime type. Try **L4** first with the low-memory preset; the docs
ask for 32 GB (A100) but `low_mem` may fit in 24 GB. If it fails, switch to A100.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print(sys.version)

## 1. Drive

Upload the `data/queries` folder from your laptop to
`MyDrive/openbind-blind-challenge/` first, using drive.google.com in a browser.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/openbind-blind-challenge')
for sub in ['predictions', 'logs', 'weights']:
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
print(sorted(p.name for p in (ROOT / 'data/queries').glob('*.json'))[:5])

## 2. Install

In [ ]:
!pip -q install openfold3 awscli

## 3. OpenBind-0 weights

`of3-ob-2025-06-30-174k.pt`: `ob` = OpenBind, and the date matches its stated
training cutoff. About 2 GB. Kept on Drive so it downloads once.

In [ ]:
CKPT = ROOT / 'weights' / 'of3-ob-2025-06-30-174k.pt'
if not CKPT.exists():
    !aws s3 cp s3://openfold3-data/openfold3-parameters/of3-ob-2025-06-30-174k.pt "{CKPT}" --no-sign-request
print(CKPT.exists(), f'{CKPT.stat().st_size/1e9:.1f} GB' if CKPT.exists() else '')

## 4. Runner config

Submission needs PDB; the default is mmCIF. `low_mem` is the preset that may let a
smaller GPU work — drop that block if you are on an A100 and want full speed.

In [ ]:
(ROOT / 'runner.yml').write_text('''output_writer_settings:
  structure_format: pdb
model_update:
  presets:
    - predict
    - low_mem
''')
print((ROOT / 'runner.yml').read_text())

## 5. Two compounds, timed

`test_2.json` is the first two queries pulled out of batch_000.

In [ ]:
import json, time, subprocess

q = json.load(open(ROOT / 'data/queries/batch_000.json'))['queries']
first2 = {k: q[k] for k in list(q)[:2]}
test = ROOT / 'data/queries/test_2.json'
test.write_text(json.dumps({'queries': first2}, indent=1))
print('testing:', list(first2))

cmd = ['run_openfold', 'predict',
       f'--query-json={test}',
       f'--output-dir={ROOT}/predictions',
       f'--inference-ckpt-path={CKPT}',
       f'--runner-yaml={ROOT}/runner.yml',
       '--num-diffusion-samples=1',
       '--use-msa-server']
t0 = time.time()
r = subprocess.run(cmd, capture_output=True, text=True)
print(f'{time.time()-t0:.0f}s for 2 compounds, returncode {r.returncode}')
print(r.stdout[-3000:])
if r.returncode: print(r.stderr[-4000:])

## 6. Do the arithmetic before going further

Per-compound time from above × 400 = total GPU hours. Check that against what your
Colab plan gives you. If it does not fit, the options in order are: keep
`num_diffusion_samples` at 1, precompute the MSA once and reuse it, or fall back to
docking for the whole set.

Also open one output PDB and confirm the ligand is actually in it.

In [ ]:
import pathlib
outs = sorted((ROOT / 'predictions').rglob('*.pdb'))
print(len(outs), 'pdb files')
for p in outs[:2]:
    txt = p.read_text()
    het = [l for l in txt.splitlines() if l.startswith('HETATM')]
    print(p.name, f'{len(txt.splitlines())} lines, {len(het)} ligand atoms')

## 7. Batch run (only after step 6 adds up)

Skips finished work, so a disconnect costs one batch at most.

In [ ]:
import json, datetime, time, subprocess, pathlib

LOG = ROOT / 'logs' / 'batches.jsonl'
done = set()
if LOG.exists():
    done = {json.loads(l)['batch'] for l in LOG.read_text().splitlines() if json.loads(l)['ok']}

batches = sorted((ROOT / 'data/queries').glob('batch_*.json'))
todo = [b for b in batches if b.stem not in done]
print(f'{len(done)} batches done, {len(todo)} to go')

for b in todo[:1]:          # raise this number once you trust the timing
    cmd = ['run_openfold', 'predict',
           f'--query-json={b}',
           f'--output-dir={ROOT}/predictions',
           f'--inference-ckpt-path={CKPT}',
           f'--runner-yaml={ROOT}/runner.yml',
           '--num-diffusion-samples=1',
           '--use-msa-server']
    t0 = time.time(); r = subprocess.run(cmd, capture_output=True, text=True)
    secs = time.time() - t0
    with LOG.open('a') as f:
        f.write(json.dumps({'batch': b.stem, 'ok': r.returncode == 0,
                            'seconds': round(secs, 1),
                            'when': datetime.datetime.now().isoformat(timespec='seconds'),
                            'error': None if r.returncode == 0 else r.stderr[-1500:]}) + '\n')
    print(f'{b.stem}: {secs/60:.1f} min, {"ok" if r.returncode==0 else "FAILED"}')